# Fabric Workspace GUID Extractor
Extracts all artifact GUIDs from one or more Microsoft Fabric workspaces
and optionally updates the Variable Library definition via the Fabric REST API.
**Usage:**
1. Set `WORKSPACE_NAMES` below to target specific workspaces by name, or leave empty to auto-detect the current workspace.
2. Fill in `ARTIFACT_MAPPING` (and optionally `REFERENCE_MAPPING` / `ARTIFACT_DETAIL_MAPPING`) with the artifacts you want variables for. Every mapping key is a `(displayName, itemType)` pair.
3. Run all cells with the shipped defaults (`UPDATE_VALUE_SETS = False`) to get the item inventory and confirm your mappings resolve.
4. Set `UPDATE_VALUE_SETS = True` and configure `VALUE_SETS_TO_UPDATE` to see the full `CHANGED` / `MATCH` / `SKIP` diff against the Variable Library. Use `"default"` to target the base variables, or specific value set names (e.g. `"Env-1D"`) for environment overrides. Nothing is written yet - `DRY_RUN` defaults to `True`.
5. Review the diff, then set `DRY_RUN = False` to post the updated definition back to Fabric.


## Configuration

Pick the workspaces to scan, name the target Variable Library, and decide which value sets get written. `"__current__"` is a convenience token that resolves to the workspace this notebook is running in - useful when the same notebook is deployed across dev/test/prod.

Both write gates ship closed: `UPDATE_VALUE_SETS = False` skips the Variable Library entirely, and `DRY_RUN = True` computes the diff without posting it. Reading the diff is the common case; writing is the deliberate second step.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────
# Add workspace names here to extract GUIDs from multiple workspaces.
# Leave empty (the shipped default) to auto-detect the current workspace via
# notebookutils - that plus UPDATE_VALUE_SETS = False is the intended first run,
# which prints the item inventory without touching anything.
WORKSPACE_NAMES = []

# --- Write safety ---
# This notebook rewrites a config artifact, so both gates below ship closed.
#
# UPDATE_VALUE_SETS  False (default) - skip the Variable Library path entirely.
#                    True  - fetch the definition and compute the diff.
# DRY_RUN            True (default)  - print the CHANGED / MATCH / SKIP diff but
#                                      never POST. This is the useful output.
#                    False - post the updated definition back to Fabric.
#
# Read the diff first, then set DRY_RUN = False as a deliberate second step.
UPDATE_VALUE_SETS = False
DRY_RUN = True

# Display name of the Variable Library item in Fabric
VARIABLE_LIBRARY_NAME = "<VariableLibraryName>"

# Which workspace contains the Variable Library to update.
# Must be one of the names in WORKSPACE_NAMES above.
VARIABLE_LIBRARY_WORKSPACE = "__current__"

# Create WorkspaceId / WorkspaceName variables when the library doesn't have them?
# Off by default - most libraries don't want them, and adding two variables churns
# the Git-integration diff for every consumer of the library. Entries that already
# exist under those names are kept up to date either way.
ADD_WORKSPACE_IDENTITY_VARS = False

# Map value set names to workspace names. Use "default" to update the base variables.
# Use "__current__" as the workspace name to auto-resolve to the workspace this notebook runs in.
# Only listed targets are touched; unlisted value sets are passed through unchanged.
VALUE_SETS_TO_UPDATE = {
    "default": "__current__",
    "<SetName1>": "<WorkspaceName>-<EnvironmentSuffix>",
    # "<SetName2>": "<WorkspaceName>-<EnvironmentSuffix>",
    # "<SetName3>": "<WorkspaceName>-<EnvironmentSuffix>",
}

# --- Strict Mode ---
# False (default) - print the summary and continue even if some value sets
#                   failed or were skipped. Right for scheduled runs where the
#                   per-set report is enough signal.
# True  - raise RuntimeError after the summary when any value set failed, or when
#         a required artifact mapping matched nothing in any scanned workspace.
#         Use when running from CI / orchestration and you want the notebook exit
#         code to reflect state.
STRICT = False

# Canonical Tier-2 result buckets - populated by the extract and update loops
# below. Each "name" is a value-set identifier (e.g. "default", "Env-1D"), a
# workspace scan label, or an "artifact-mapping:..." entry for a configured
# mapping that matched no item.
results = {
    "succeeded": [],  # list[str]
    "skipped":   [],  # list[dict]: {"name": str, "reason": str}
    "failed":    [],  # list[dict]: {"name": str, "error": str}
}

## Authentication & Workspace Resolution

Acquires a Fabric token from the notebook runtime and defines the shared request layer every later call goes through: `fabric_request` (429 / transient-5xx back-off honoring `Retry-After`, plus a one-shot token refresh on 401), `get_paginated` (follows `continuationUri` to the end of a list endpoint), and `poll_lro` (polls a long-running operation to completion under a hard deadline).

In [ ]:
import base64
import json
import re
import time
from typing import Any, Dict, List, Optional, Set, Tuple

import requests

# ── Authenticate via Fabric runtime ──────────────────────────────────────────────
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
FABRIC_TOKEN_SCOPE = "https://api.fabric.microsoft.com/.default"

# Retry / timeout budget. A wide multi-workspace scan plus one detail GET per
# matched item is exactly the shape that trips Fabric throttling, so every call
# goes through fabric_request() below rather than requests.* directly.
MAX_RETRIES = 5            # per request, for 429 and transient 5xx
DEFAULT_RETRY_AFTER = 5    # seconds, when Retry-After is absent or unparseable
LRO_TIMEOUT_SECONDS = 900  # hard deadline for a single long-running operation

TOKEN = ""
HEADERS: Dict[str, str] = {}


def refresh_token() -> None:
    """Acquire a Fabric API token and rebuild the shared request headers.

    :returns: None. Mutates the module-level TOKEN and HEADERS.
    """
    global TOKEN, HEADERS
    TOKEN = notebookutils.credentials.getToken(FABRIC_TOKEN_SCOPE)
    HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


refresh_token()


def parse_retry_after(response: requests.Response, default: int = DEFAULT_RETRY_AFTER) -> int:
    """Read the Retry-After header as a delay in seconds.

    RFC 9110 permits an HTTP-date instead of a delay-seconds integer, so a bare
    int() would raise on a legal response. Fall back to the default instead.

    :param response: Response whose headers to inspect.
    :param default: Seconds to use when the header is missing or non-integer.
    :returns: A positive number of seconds to wait.
    """
    raw = response.headers.get("Retry-After")
    if raw is None:
        return default
    try:
        return max(1, int(raw))
    except (TypeError, ValueError):
        return default


def fabric_request(method: str, url: str, json_body: Optional[Dict] = None,
                   max_retries: int = MAX_RETRIES) -> requests.Response:
    """Issue a Fabric REST call with throttling back-off and one token refresh.

    Retries 429 and transient 5xx responses honoring Retry-After, and refetches
    the token once on a 401 - a wide scan plus LRO polling can outlive the token
    acquired at notebook start. Non-retriable responses are returned as-is for
    the caller to interpret.

    :param method: HTTP verb, e.g. "GET" or "POST".
    :param url: Absolute request URL.
    :param json_body: Optional JSON body to send.
    :param max_retries: Maximum retry attempts for throttled / transient failures.
    :returns: The final requests.Response, successful or not.
    """
    response = None
    refreshed = False

    for attempt in range(max_retries + 1):
        response = requests.request(method, url, headers=HEADERS, json=json_body)

        if response.status_code == 401 and not refreshed:
            print("  401 Unauthorized - refreshing token and retrying once...")
            refresh_token()
            refreshed = True
            continue

        if response.status_code == 429 or response.status_code in (502, 503, 504):
            if attempt == max_retries:
                break
            wait = parse_retry_after(response)
            print(
                f"  {response.status_code} from {url} - retrying in {wait}s "
                f"(attempt {attempt + 1}/{max_retries})"
            )
            time.sleep(wait)
            continue

        break

    return response


def get_paginated(url: str, what: str) -> List[Dict]:
    """Follow continuationUri to the end of a Fabric list endpoint.

    :param url: First-page URL.
    :param what: Noun used in the error message, e.g. "workspaces".
    :returns: Concatenated `value` arrays from every page.
    :raises RuntimeError: If any page returns a non-200 status.
    """
    collected: List[Dict] = []

    while url:
        response = fabric_request("GET", url)
        if response.status_code != 200:
            raise RuntimeError(
                f"Failed to list {what}: {response.status_code} {response.text}"
            )
        data = response.json()
        collected.extend(data.get("value", []))
        url = data.get("continuationUri")

    return collected


def get_workspace_display_name(workspace_id: str) -> Optional[str]:
    """Look up a workspace's display name by GUID via the Get Workspace API.

    :param workspace_id: Workspace GUID.
    :returns: The display name, or None if the call failed or carried no name.
    """
    response = fabric_request("GET", f"{FABRIC_API_BASE}/workspaces/{workspace_id}")
    if response.status_code != 200:
        print(
            f"  WARNING: could not read workspace {workspace_id}: "
            f"{response.status_code} {response.text}"
        )
        return None
    return response.json().get("displayName")


def resolve_workspace_names(names: List[str]) -> List[Tuple[str, str]]:
    """Resolve workspace display names to (workspace_id, workspace_name) pairs.

    :param names: Workspace display names to resolve.
    :returns: One (id, name) pair per input name, in input order.
    :raises RuntimeError: If any name is not among the accessible workspaces.
    """
    all_workspaces = get_paginated(f"{FABRIC_API_BASE}/workspaces", "workspaces")
    name_lookup = {ws["displayName"]: ws["id"] for ws in all_workspaces}

    resolved = []
    for name in names:
        ws_id = name_lookup.get(name)
        if not ws_id:
            print(f"WARNING: Workspace '{name}' not found. Available workspaces:")
            for available in sorted(name_lookup.keys()):
                print(f"  - {available}")
            raise RuntimeError(f"Workspace '{name}' not found.")
        resolved.append((ws_id, name))

    return resolved


def poll_lro(response: requests.Response,
             timeout_seconds: int = LRO_TIMEOUT_SECONDS) -> requests.Response:
    """Poll a Fabric long-running operation to completion, then fetch its result.

    :param response: The initiating response. Returned unchanged unless it is a 202.
    :param timeout_seconds: Deadline for the whole poll loop.
    :returns: The /result response on success, or the last status response otherwise.
    :raises RuntimeError: If the 202 carries no Location header to poll.
    :raises TimeoutError: If the operation is still running at the deadline.
    """
    if response.status_code != 202:
        return response

    location = response.headers.get("Location")
    if not location:
        raise RuntimeError("Operation returned 202 with no Location header - cannot poll.")

    deadline = time.monotonic() + timeout_seconds
    status = ""

    while True:
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            raise TimeoutError(
                f"Long-running operation did not complete within {timeout_seconds}s "
                f"(last status: {status or 'unknown'})."
            )

        wait = min(parse_retry_after(response), remaining)
        print(f"  Operation in progress, retrying in {wait:.0f}s...")
        time.sleep(wait)

        response = fabric_request("GET", location)
        try:
            status = response.json().get("status", "")
        except ValueError:
            status = ""

        if status not in ("NotStarted", "Running"):
            break

    if status != "Succeeded":
        return response

    # Fetch the actual result from the /result endpoint
    return fabric_request("GET", f"{location}/result")


# ── Resolve current workspace ────────────────────────────────────────────────────
# currentWorkspaceId is the documented public API; the shorter key is undocumented
# and only kept as a fallback.
ctx = notebookutils.runtime.context
CURRENT_WORKSPACE_ID = ctx.get("currentWorkspaceId") or ctx.get("workspaceId")
if not CURRENT_WORKSPACE_ID:
    raise RuntimeError("Could not detect workspace ID from notebookutils.runtime.context")

# The display name is what "__current__" resolves to, and every VALUE_SETS_TO_UPDATE
# lookup is keyed by it. The runtime context does not reliably carry it, and when it
# does it can be stale after a rename - so ask the API, which is authoritative, and
# fall back to the context only if that call fails. A sentinel like "Unknown" here
# would silently skip every "__current__" target instead of failing.
CURRENT_WORKSPACE_NAME = (
    get_workspace_display_name(CURRENT_WORKSPACE_ID)
    or ctx.get("currentWorkspaceName")
    or ctx.get("workspaceName")
)
if not CURRENT_WORKSPACE_NAME:
    raise RuntimeError(
        f"Could not resolve a display name for the current workspace "
        f"({CURRENT_WORKSPACE_ID}). '__current__' cannot be resolved without it - "
        f"name the workspace explicitly in VALUE_SETS_TO_UPDATE instead."
    )
print(f"Current workspace: {CURRENT_WORKSPACE_NAME} ({CURRENT_WORKSPACE_ID})")

# ── Resolve workspaces ───────────────────────────────────────────────────────────
if WORKSPACE_NAMES:
    WORKSPACES = resolve_workspace_names(WORKSPACE_NAMES)
else:
    WORKSPACES = [(CURRENT_WORKSPACE_ID, CURRENT_WORKSPACE_NAME)]

# Ensure current workspace is in the list (needed for "__current__" resolution)
current_in_list = any(ws_id == CURRENT_WORKSPACE_ID for ws_id, _ in WORKSPACES)
if not current_in_list:
    WORKSPACES.append((CURRENT_WORKSPACE_ID, CURRENT_WORKSPACE_NAME))

# Resolve "__current__" in VALUE_SETS_TO_UPDATE and VARIABLE_LIBRARY_WORKSPACE
VALUE_SETS_TO_UPDATE = {
    k: (CURRENT_WORKSPACE_NAME if v == "__current__" else v)
    for k, v in VALUE_SETS_TO_UPDATE.items()
}
if VARIABLE_LIBRARY_WORKSPACE == "__current__":
    VARIABLE_LIBRARY_WORKSPACE = CURRENT_WORKSPACE_NAME

print(f"Targeting {len(WORKSPACES)} workspace(s):")
for ws_id, ws_name in WORKSPACES:
    print(f"  {ws_name} -> {ws_id}")

## Artifact Mapping

Three mappings drive what ends up in the Variable Library. All of them key on **`(displayName, itemType)`** - the display name alone is ambiguous, because a Lakehouse and its auto-generated SQLEndpoint share one.

| Mapping | Produces | Use When |
|---|---|---|
| **`ARTIFACT_MAPPING`** | Scalar GUID variables - `SalesLakehouseId = "abc-123"` | You want a plain item GUID. |
| **`REFERENCE_MAPPING`** | Compound Item Reference variables - `SalesLakehouseRef = {"itemId": "abc-123", "workspaceId": "def-456"}` | You want to bind a pipeline activity or notebook input to a Fabric item directly. Fabric's "Item Reference" variable type expects the `{itemId, workspaceId}` shape, which is tedious to maintain by hand across environments. |
| **`ARTIFACT_DETAIL_MAPPING`** | Per-item metadata variables - `SalesLakehouseSqlConnectionString = "x.datawarehouse.fabric.microsoft.com"` | You need values that only the type-specific GET endpoint returns (SQL connection strings, KQL query URIs, database names, server FQDNs). Triggers one extra REST call per matched item. |

`REFERENCE_MAPPING` does NOT read the Fabric API separately - it **reuses** a GUID already extracted via `ARTIFACT_MAPPING`. So to produce `SalesLakehouseRef`, you must also have a mapping that yields `SalesLakehouseId`.

Because that scalar is often only wanted as an ingredient, **any name on the right of `REFERENCE_MAPPING` is treated as an internal helper**: it is used to build the compound value and is never appended to the library as a standalone variable. Name those helpers with a leading underscore (`_SalesLakehouseId`) to make the intent obvious. If you want both the scalar and the reference as real library variables, create the scalar in the library once - existing entries are always kept up to date, only *creation* is suppressed.

`ARTIFACT_DETAIL_MAPPING` **does** call the Fabric API again - one GET per matched item against the type-specific endpoint (e.g. `/lakehouses/{id}`) - because the workspace `/items` listing only returns identity fields. `ITEM_TYPE_URL_SEGMENT` lists the item types Microsoft documents as returning extra properties; an item type outside that list prints a warning rather than silently yielding nothing.

### Reporting unmatched mappings

After the scan, every configured `(displayName, itemType)` that matched no item in **any** scanned workspace is reported as a failure - that is how a typo or a renamed item surfaces instead of quietly doing nothing. Add the key to `OPTIONAL_ARTIFACTS` to downgrade it to an informational note. The rule:

- **Remove the mapping** when the variable should point somewhere this notebook has no business managing (another team's workspace, a hand-maintained value).
- **Mark it optional** when the artifact is merely absent from this workspace but the mapping is still correct where it does exist.

### Not covered: `ConnectionReference` variables

Connections are tenant-scoped and are not returned by the workspace `/items` listing, so they cannot be extracted by this notebook's model. `ConnectionReference` variables have to be set another way; the type is recognized for validation only.

Typical pattern: populate `ARTIFACT_MAPPING` for every item you need, then add `REFERENCE_MAPPING` entries only for items consumed by pipelines/notebooks that require the compound form, and `ARTIFACT_DETAIL_MAPPING` only for items whose connection strings or other deep fields you need to surface as variables.

In [ ]:
# Mapping of Fabric artifacts to variables.json variable names.
# Keyed by (displayName, itemType) -> variableName.
#
# The item type is mandatory, not optional. A display name alone is ambiguous:
# every Lakehouse surfaces as TWO items - the Lakehouse and its auto-generated
# SQLEndpoint - which share a display name but have different GUIDs. Keying on
# the name alone binds whichever the API happened to return first, so dev could
# bind the Lakehouse and prod the SQL endpoint from byte-identical config.
#
# Worked examples:
#   ("Sales_Lakehouse", "Lakehouse")   -> "SalesLakehouseId"
#   ("Sales_Lakehouse", "SQLEndpoint") -> "SalesSqlEndpointId"
#   ("Sales_Warehouse", "Warehouse")   -> "SalesWarehouseId"
#
# Naming convention: prefix a variable with an underscore when it exists only to
# feed REFERENCE_MAPPING and should never become a library variable of its own
# (e.g. "_SalesLakehouseId"). See the REFERENCE_MAPPING notes below.
ARTIFACT_MAPPING = {
    # ("<LakehouseDisplayName>", "Lakehouse"):    "<LakehouseIdVar>",
    # ("<LakehouseDisplayName>", "SQLEndpoint"):  "<SqlEndpointIdVar>",
    # ("<WarehouseDisplayName>", "Warehouse"):    "<WarehouseIdVar>",
    # ("<PipelineDisplayName>",  "DataPipeline"): "<PipelineIdVar>",
    # ("<NotebookDisplayName>",  "Notebook"):     "<NotebookIdVar>",
}

# Artifacts that may legitimately be missing from a scanned workspace.
# Keys use the same (displayName, itemType) shape as ARTIFACT_MAPPING and
# ARTIFACT_DETAIL_MAPPING.
#
# The distinction this encodes:
#   - Remove the mapping entirely when the variable should point somewhere this
#     notebook has no business managing (another team's workspace, a manual value).
#   - Mark it optional when the artifact is merely absent from this workspace but
#     the mapping is still correct where it does exist.
# Anything configured, unmatched, and NOT listed here is reported as a failure -
# that is how a typo or a renamed item surfaces instead of silently doing nothing.
OPTIONAL_ARTIFACTS = {
    # ("<EventstreamDisplayName>", "Eventstream"),
    # ("<KqlDatabaseDisplayName>", "KQLDatabase"),
}

# Mapping of ItemReference variable names to their corresponding GUID variable names.
# Used to auto-populate {"itemId": <guid>, "workspaceId": <ws_id>} in value sets.
#
# Worked example:
#   "SalesLakehouseRef" -> "_SalesLakehouseId" tells the extractor: create a variable
#   called SalesLakehouseRef whose value is the compound object
#       {"itemId": "<_SalesLakehouseId guid>", "workspaceId": "<current workspace guid>"}
#   This is the shape Fabric's "Item Reference" variable type expects - pipelines and
#   notebooks can then bind directly to SalesLakehouseRef instead of juggling two
#   separate variables for itemId and workspaceId.
#
# Any name appearing on the RIGHT of this map is treated as an internal helper: it
# is consumed to build the compound value and is never appended to the library as a
# standalone variable. If you want the scalar GUID as its own library variable too,
# create it in the library once (or map the same artifact to a second variable name)
# - an entry that already exists is still kept up to date.
REFERENCE_MAPPING = {
    # "<LakehouseRefVar>": "<LakehouseIdVar>",
    # "<WarehouseRefVar>": "<WarehouseIdVar>",
    # "<PipelineRefVar>":  "<PipelineIdVar>",
}

# Optional per-item metadata fields fetched via the type-specific GET endpoint.
# Keyed by (displayName, itemType) -> {variableName: dotted JSON path in the response}.
# Values are stamped into the variable library alongside the GUIDs from ARTIFACT_MAPPING.
#
# Worked example:
#   ("Sales_Lakehouse", "Lakehouse") with
#       {"SalesLakehouseSqlConnectionString": "properties.sqlEndpointProperties.connectionString"}
#   tells the extractor: after resolving the GUID for the Lakehouse named
#   "Sales_Lakehouse", also GET /workspaces/{wsId}/lakehouses/{id}, walk
#   `properties.sqlEndpointProperties.connectionString`, and stamp the result into
#   the variable library as SalesLakehouseSqlConnectionString.
ARTIFACT_DETAIL_MAPPING = {
    # ("<LakehouseDisplayName>", "Lakehouse"): {
    #     "<LakehouseSqlConnectionStringVar>": "properties.sqlEndpointProperties.connectionString",
    #     "<LakehouseSqlEndpointIdVar>":       "properties.sqlEndpointProperties.id",
    # },
    # ("<SqlDatabaseDisplayName>", "SQLDatabase"): {
    #     "<DatabaseNameVar>":              "properties.databaseName",
    #     "<DatabaseSqlEndpointStringVar>": "properties.serverFqdn",
    # },
    # ("<WarehouseDisplayName>", "Warehouse"): {
    #     "<WarehouseSqlConnectionStringVar>": "properties.connectionString",
    # },
    # ("<KqlDatabaseDisplayName>", "KQLDatabase"): {
    #     "<KqlQueryUriVar>": "properties.queryServiceUri",
    # },
}

# URL segment under /workspaces/{wsId}/ for the per-itemType GET endpoint.
# These are the item types Microsoft documents as returning properties beyond the
# generic Get Item response ("Get type-specific item properties" in the OneLake
# catalog REST article). Every other type returns the generic shape, so a detail
# mapping against one would only surface fields the /items listing already has.
ITEM_TYPE_URL_SEGMENT = {
    "Environment":        "environments",
    "Eventhouse":         "eventhouses",
    "KQLDatabase":        "kqlDatabases",
    "Lakehouse":          "lakehouses",
    "MirroredDatabase":   "mirroredDatabases",
    "MLExperiment":       "mlExperiments",
    "SparkJobDefinition": "sparkJobDefinitions",
    "SQLDatabase":        "sqlDatabases",
    "Warehouse":          "warehouses",
}


def get_workspace_items(workspace_id: str) -> List[Dict]:
    """Fetch every item in a workspace, following pagination.

    Raises rather than returning an empty list on failure: an empty mapping is
    indistinguishable from a clean no-op downstream, so a 403 or a throttled
    request would otherwise be reported as "already up to date".

    :param workspace_id: Workspace GUID to enumerate.
    :returns: All item objects in the workspace.
    :raises RuntimeError: If the listing call fails.
    """
    return get_paginated(
        f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items",
        f"items for workspace {workspace_id}",
    )


def fetch_item_details(workspace_id: str, item_id: str, item_type: str) -> Optional[Dict]:
    """Fetch an item via its type-specific GET endpoint.

    :param workspace_id: Workspace GUID containing the item.
    :param item_id: Item GUID.
    :param item_type: Fabric item type, e.g. "Lakehouse".
    :returns: The parsed response, or None if the type has no detail endpoint or
        the call failed. Both cases print a warning.
    """
    url_segment = ITEM_TYPE_URL_SEGMENT.get(item_type)
    if not url_segment:
        print(
            f"  WARNING: no detail endpoint configured for item type '{item_type}'. "
            f"Add it to ITEM_TYPE_URL_SEGMENT if Fabric documents one."
        )
        return None

    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/{url_segment}/{item_id}"
    response = fabric_request("GET", url)
    if response.status_code != 200:
        print(
            f"  WARNING: Failed to fetch {url_segment}/{item_id}: "
            f"{response.status_code} {response.text}"
        )
        return None

    return response.json()


def get_nested(data: Dict, dotted_path: str) -> Any:
    """Walk a dotted JSON path on a dict.

    :param data: Object to walk.
    :param dotted_path: Path such as "properties.sqlEndpointProperties.id".
    :returns: The value at the path, or None if any segment is missing.
    """
    cursor = data
    for key in dotted_path.split("."):
        if not isinstance(cursor, dict):
            return None
        cursor = cursor.get(key)
    return cursor


def extract_guids(
    workspace_id: str, workspace_name: str = ""
) -> Tuple[Dict[str, Any], Optional[str], Set[Tuple[str, str]]]:
    """Extract configured GUIDs and detail values from a single workspace.

    :param workspace_id: Workspace GUID to scan.
    :param workspace_name: Display name, used only for log output.
    :returns: (guid_mapping, variable_library_id, matched_keys) where matched_keys
        holds the (displayName, itemType) pairs that resolved to a real item.
    :raises RuntimeError: If the workspace item listing fails.
    """
    items = get_workspace_items(workspace_id)
    guid_mapping: Dict[str, Any] = {}
    matched_keys: Set[Tuple[str, str]] = set()
    variable_library_id = None

    label = f"{workspace_name} ({workspace_id})" if workspace_name else workspace_id
    print(f"\n{label} - {len(items)} items")
    print(f"{'Display Name':<45} {'Type':<20} {'ID'}")
    print("-" * 110)

    for item in sorted(items, key=lambda x: x.get("displayName", "")):
        display_name = item.get("displayName", "")
        item_id = item.get("id", "")
        item_type = item.get("type", "")
        key = (display_name, item_type)

        print(f"{display_name:<45} {item_type:<20} {item_id}")

        if key in ARTIFACT_MAPPING:
            guid_mapping[ARTIFACT_MAPPING[key]] = item_id
            matched_keys.add(key)

        # Truthy check skips null/empty values so an unprovisioned SQL endpoint
        # doesn't overwrite a valid one stamped on a previous run.
        if key in ARTIFACT_DETAIL_MAPPING:
            details = fetch_item_details(workspace_id, item_id, item_type)
            if details:
                matched_keys.add(key)
                for var_name, json_path in ARTIFACT_DETAIL_MAPPING[key].items():
                    value = get_nested(details, json_path)
                    if value:
                        guid_mapping[var_name] = value
                    else:
                        print(
                            f"  WARNING: {display_name} ({item_type}) has no value at "
                            f"'{json_path}' - leaving {var_name} untouched."
                        )

        if display_name == VARIABLE_LIBRARY_NAME and item_type == "VariableLibrary":
            variable_library_id = item_id

    return guid_mapping, variable_library_id, matched_keys

## Extract GUIDs

Runs `extract_guids` for every workspace in scope, prints a per-workspace table of every item (display name, type, GUID), and captures the Variable Library's own GUID from the workspace named in `VARIABLE_LIBRARY_WORKSPACE`.

In [ ]:
# ── Extract GUIDs from all workspaces ────────────────────────────────────────────
all_guid_mappings = {}  # {workspace_id: {var_name: value}}
variable_library_info = None  # (workspace_id, item_id, workspace_name)
matched_across_workspaces = set()  # {(displayName, itemType)}

for ws_id, ws_name in WORKSPACES:
    mapping, vl_id, matched = extract_guids(ws_id, ws_name)
    all_guid_mappings[ws_id] = mapping
    matched_across_workspaces |= matched

    # Only use the VL from the explicitly configured workspace
    if vl_id and ws_name == VARIABLE_LIBRARY_WORKSPACE:
        variable_library_info = (ws_id, vl_id, ws_name)

    print(f"\n{'='*110}")
    print(f"GUID MAPPING for {ws_name} ({ws_id}):")
    print(f"{'='*110}")
    for var_name, value in mapping.items():
        print(f"  {var_name}: {value}")

# ── Report configured mappings that matched nothing ──────────────────────────────
# Aggregated across every scanned workspace: with several workspaces in scope most
# mappings only match in one of them, so a per-workspace check would be all noise.
# A key that matched nowhere is a typo, a renamed item, or a genuinely absent
# artifact - the first two are config bugs, the third is what OPTIONAL_ARTIFACTS
# is for.
configured_keys = set(ARTIFACT_MAPPING) | set(ARTIFACT_DETAIL_MAPPING)
unmatched = configured_keys - matched_across_workspaces
unmatched_required = sorted(unmatched - OPTIONAL_ARTIFACTS)
unmatched_optional = sorted(unmatched & OPTIONAL_ARTIFACTS)

if unmatched_optional:
    print(f"\nOptional artifacts not present in any scanned workspace ({len(unmatched_optional)}):")
    for display_name, item_type in unmatched_optional:
        print(f"  note: '{display_name}' ({item_type}) - listed in OPTIONAL_ARTIFACTS")

if unmatched_required:
    print(f"\nWARNING: {len(unmatched_required)} configured mapping(s) matched no item:")
    for display_name, item_type in unmatched_required:
        error = (
            f"'{display_name}' ({item_type}) not found in any scanned workspace. "
            f"Check the display name and item type, or add it to OPTIONAL_ARTIFACTS "
            f"if the artifact is deliberately absent here."
        )
        print(f"  {error}")
        results["failed"].append({
            "name": f"artifact-mapping:{display_name}/{item_type}",
            "error": error,
        })

if variable_library_info:
    vl_ws_id, vl_item_id, vl_ws_name = variable_library_info
    print(
        f"\nVariable Library '{VARIABLE_LIBRARY_NAME}' found in {vl_ws_name} ({vl_item_id})"
    )
else:
    # Track as a structured failure - this blocks the update flow even though
    # GUID extraction itself succeeded. The update cell below will check and
    # skip with a matching entry; surfacing it here gives the summary context.
    reason = (
        f"Variable Library '{VARIABLE_LIBRARY_NAME}' not found in workspace "
        f"'{VARIABLE_LIBRARY_WORKSPACE}'. Check VARIABLE_LIBRARY_WORKSPACE config."
    )
    print(f"\nWARNING: {reason}")
    results["failed"].append({"name": "variable-library-lookup", "error": reason})

## Update Variable Library

When `UPDATE_VALUE_SETS = True`, fetches the Variable Library definition via `getDefinition`, applies the extracted GUIDs to `variables.json` (the `default` value set) and any targeted `valueSets/<name>.json` overrides, and prints the diff. With `DRY_RUN = False` it then posts the full definition back via `updateDefinition`. Every part of the original definition is re-sent - the Fabric API replaces the entire definition on update, so omitting any part would delete it.

Entries added to `variables.json` are built with a `type` (required by the schema and inferred from the value's shape); value-set overrides carry only `name`/`value`. Before any value is written it is checked against the variable's declared type, so a bare GUID can't land in a variable declared `ItemReference`. Parts with no changes are passed through byte-for-byte rather than reserialized, to keep the Git-integration diff clean.

In [ ]:
# ── Update Variable Library via Fabric REST API (optional) ────────────────────────
GUID_PATTERN = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.IGNORECASE
)

# Shape checks per declared Fabric variable type, keyed lowercase because Fabric
# serializes ItemReference title-cased and connectionReference camel-cased.
# bool is excluded from the numeric checks - it is an int subclass in Python.
VALUE_TYPE_VALIDATORS = {
    "boolean":             lambda v: isinstance(v, bool),
    "datetime":            lambda v: isinstance(v, str),
    "guid":                lambda v: isinstance(v, str) and bool(GUID_PATTERN.match(v)),
    "integer":             lambda v: isinstance(v, int) and not isinstance(v, bool),
    "number":              lambda v: isinstance(v, (int, float)) and not isinstance(v, bool),
    "string":              lambda v: isinstance(v, str),
    "itemreference":       lambda v: isinstance(v, dict) and {"itemId", "workspaceId"} <= set(v),
    "connectionreference": lambda v: isinstance(v, dict) and "connectionId" in v,
}


def infer_variable_type(value: Any) -> str:
    """Infer the Fabric variable type for a value being added to a library.

    :param value: The value to classify.
    :returns: A Fabric variable type name, defaulting to "String".
    """
    if isinstance(value, dict):
        if "itemId" in value:
            return "ItemReference"
        if "connectionId" in value:
            return "ConnectionReference"
    if isinstance(value, bool):
        return "Boolean"
    if isinstance(value, int):
        return "Integer"
    if isinstance(value, float):
        return "Number"
    if isinstance(value, str) and GUID_PATTERN.match(value):
        return "Guid"
    return "String"


def make_variable_entry(name: str, value: Any, is_default: bool) -> Dict[str, Any]:
    """Build a schema-valid variable entry.

    variables.json entries carry name/note/type/value - `type` is required by the
    schema, so an entry without it is rejected. Value-set overrides carry only
    name/value. Field order matches Fabric's own serialization to keep the
    Git-integration diff minimal.

    :param name: Variable name.
    :param value: Variable value.
    :param is_default: True for variables.json, False for a value-set override.
    :returns: The entry dict.
    """
    if not is_default:
        return {"name": name, "value": value}
    return {"name": name, "note": "", "type": infer_variable_type(value), "value": value}


def validate_value_type(var_name: str, declared_type: Optional[str], value: Any) -> None:
    """Check a computed value's shape against the variable's declared type.

    Catches the common config mistake of listing an ItemReference variable in
    ARTIFACT_MAPPING (which yields a bare GUID string) instead of
    REFERENCE_MAPPING (which yields the compound object), before it is posted.

    :param var_name: Variable name, for the error message.
    :param declared_type: Type from variables.json, or None when unknown.
    :param value: The value about to be written.
    :returns: None.
    :raises ValueError: If the value's shape contradicts the declared type.
    """
    if not declared_type:
        return

    validator = VALUE_TYPE_VALIDATORS.get(declared_type.lower())
    if validator is None:
        # A type Fabric added after this notebook was written - don't block on it.
        print(f"    NOTE    {var_name}: unrecognized type '{declared_type}', shape not checked")
        return

    if not validator(value):
        raise ValueError(
            f"'{var_name}' is declared '{declared_type}' but the extracted value is "
            f"{type(value).__name__} {value!r}. Check whether it belongs in "
            f"ARTIFACT_MAPPING (scalar GUID) or REFERENCE_MAPPING (compound reference)."
        )


def apply_guid_updates(variables: List[Dict], ws_id: str, ws_name: str,
                       guid_mapping: Dict[str, Any], label: str,
                       is_default: bool = False,
                       declared_types: Optional[Dict[str, str]] = None) -> List[str]:
    """Apply extracted GUIDs to a list of variable entries in place.

    Works for both variables.json entries (fields: name/note/type/value) and
    value-set override entries (name/value). This is a pure in-place mutation
    helper - orchestration is in the caller below.

    :param variables: The mutable list of variable entries to update.
    :param ws_id: Workspace GUID to pair with WorkspaceId / ItemReference values.
    :param ws_name: Workspace display name to pair with WorkspaceName values.
    :param guid_mapping: Extracted {var_name: value} map for the target workspace.
    :param label: Descriptive label for log output (e.g. "default (variables.json)").
    :param is_default: True when writing variables.json, False for a value set.
    :param declared_types: {var_name: type} from variables.json, used to validate
        value-set entries, which don't carry their own type.
    :returns: List of variable names that were updated or added.
    :raises ValueError: If a computed value contradicts the variable's declared type.
    """
    updates_made = []
    declared_types = declared_types or {}
    existing_names = {v["name"] for v in variables}

    # Names on the right of REFERENCE_MAPPING exist only to build compound values.
    # They are still updated if the library already has them, but are never created.
    reference_sources = set(REFERENCE_MAPPING.values())

    print(f"\n  {label}:")
    print(f"    Extracted GUIDs ({len(guid_mapping)}): {list(guid_mapping.keys())}")
    print(f"    Existing variables ({len(existing_names)}): {sorted(existing_names)}")

    def append_entry(name: str, value: Any) -> None:
        """Append one new entry and record it, keeping variable names unique."""
        variables.append(make_variable_entry(name, value, is_default))
        existing_names.add(name)
        updates_made.append(name)
        print(f"    ADDED   {name}: {value}")

    for entry in variables:
        var_name = entry.get("name")
        old_value = entry.get("value")
        new_value = None

        if var_name == "WorkspaceId":
            new_value = ws_id
        elif var_name == "WorkspaceName":
            new_value = ws_name
        elif var_name in guid_mapping:
            new_value = guid_mapping[var_name]
        elif var_name in REFERENCE_MAPPING:
            item_guid_var = REFERENCE_MAPPING[var_name]
            if item_guid_var in guid_mapping:
                new_value = {"itemId": guid_mapping[item_guid_var], "workspaceId": ws_id}

        if new_value is None:
            print(f"    SKIP    {var_name}: not in extracted GUIDs")
            continue

        validate_value_type(var_name, entry.get("type") or declared_types.get(var_name), new_value)

        if old_value != new_value:
            entry["value"] = new_value
            updates_made.append(var_name)
            print(f"    CHANGED {var_name}: {old_value} -> {new_value}")
        else:
            print(f"    MATCH   {var_name}: {old_value}")

    # Add missing workspace identity entries (opt-in - see ADD_WORKSPACE_IDENTITY_VARS)
    if ADD_WORKSPACE_IDENTITY_VARS:
        for name, value in (("WorkspaceId", ws_id), ("WorkspaceName", ws_name)):
            if name not in existing_names:
                append_entry(name, value)

    # Add missing entries for extracted values not already present
    for var_name, value in guid_mapping.items():
        if var_name in existing_names or var_name in reference_sources:
            continue
        append_entry(var_name, value)

    for ref_name, guid_var in REFERENCE_MAPPING.items():
        if ref_name in existing_names or guid_var not in guid_mapping:
            continue
        append_entry(ref_name, {"itemId": guid_mapping[guid_var], "workspaceId": ws_id})

    if updates_made:
        print(f"    >> {len(updates_made)} variable(s) updated/added")
    else:
        print("    >> already up to date")

    return updates_made


def collect_declared_types(definition: Dict) -> Dict[str, str]:
    """Read the declared type of every variable from the definition's variables.json.

    Value-set overrides carry only name/value, so their declared type has to come
    from the default part. Pre-scanned rather than picked up mid-loop because part
    order is not guaranteed.

    :param definition: The `definition` object from getDefinition.
    :returns: {variable_name: declared_type}, empty if variables.json is unreadable.
    """
    declared: Dict[str, str] = {}

    for part in definition["parts"]:
        if part["path"] != "variables.json":
            continue
        try:
            payload = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
        except ValueError as ex:
            print(f"WARNING: could not parse variables.json for type validation: {ex}")
            continue
        for entry in payload.get("variables", []):
            if entry.get("name") and entry.get("type"):
                declared[entry["name"]] = entry["type"]

    return declared


if not UPDATE_VALUE_SETS:
    # User explicitly opted out - not a failure, just nothing to do.
    print("\nSkipping value set updates (set UPDATE_VALUE_SETS = True to enable).")
    for vs_name in VALUE_SETS_TO_UPDATE:
        results["skipped"].append({"name": vs_name, "reason": "UPDATE_VALUE_SETS=False"})

elif not variable_library_info:
    # The extract cell already recorded the VL-lookup failure. Record each
    # targeted value-set as skipped so the summary shows what was attempted.
    print(
        f"ERROR: Cannot update - Variable Library '{VARIABLE_LIBRARY_NAME}' not found."
    )
    for vs_name in VALUE_SETS_TO_UPDATE:
        results["skipped"].append({"name": vs_name, "reason": "variable library not found"})

elif not VALUE_SETS_TO_UPDATE:
    # Empty target map - no work defined. Also not a failure.
    print("No targets in VALUE_SETS_TO_UPDATE - nothing to do.")

else:
    vl_ws_id, vl_item_id, vl_ws_name = variable_library_info
    print(f"Fetching definition for '{VARIABLE_LIBRARY_NAME}' from {vl_ws_name}...")

    # GET current Variable Library definition
    get_url = (
        f"{FABRIC_API_BASE}/workspaces/{vl_ws_id}"
        f"/VariableLibraries/{vl_item_id}/getDefinition"
    )
    response = poll_lro(fabric_request("POST", get_url))

    if response.status_code != 200:
        # Tier-1 condition - can't proceed without the definition. Raise
        # regardless of STRICT; there is no per-value-set outcome to collect.
        raise RuntimeError(
            f"Failed to get definition for '{VARIABLE_LIBRARY_NAME}': "
            f"{response.status_code} {response.text}"
        )

    definition = response.json()["definition"]
    declared_types = collect_declared_types(definition)

    # Build workspace name -> (ws_id, guid_mapping) lookup
    ws_lookup = {
        ws_name: (ws_id, all_guid_mappings.get(ws_id, {}))
        for ws_id, ws_name in WORKSPACES
    }

    # Validate all targets in VALUE_SETS_TO_UPDATE reference known workspaces.
    # Any target pointing at an unknown workspace is skipped - we cannot
    # produce GUID values for it without having scanned it.
    for target, ws_name in VALUE_SETS_TO_UPDATE.items():
        if ws_name not in ws_lookup:
            reason = f"workspace '{ws_name}' not in WORKSPACE_NAMES"
            print(
                f"WARNING: VALUE_SETS_TO_UPDATE['{target}'] references "
                f"{reason}. Skipping."
            )
            results["skipped"].append({"name": target, "reason": reason})

    updated_parts = []
    touched_targets = set()  # value-set names that matched a part in the definition
    pending_changes = 0      # changes computed but withheld by DRY_RUN

    for part in definition["parts"]:
        path = part["path"]
        raw = base64.b64decode(part["payload"]).decode("utf-8")

        target_key = None
        if path == "variables.json" and "default" in VALUE_SETS_TO_UPDATE:
            target_key = "default"
        elif path.startswith("valueSets/"):
            # Extract value set name from path (e.g. "valueSets/Env-1D.json" -> "Env-1D")
            vs_name = path.split("/")[-1].replace(".json", "")
            if vs_name in VALUE_SETS_TO_UPDATE:
                target_key = vs_name

        if target_key is not None:
            touched_targets.add(target_key)
            target_ws_name = VALUE_SETS_TO_UPDATE[target_key]
            if target_ws_name in ws_lookup:
                ws_id, guid_mapping = ws_lookup[target_ws_name]
                try:
                    data = json.loads(raw)
                    is_default = path == "variables.json"

                    if is_default:
                        updates = apply_guid_updates(
                            data["variables"], ws_id, target_ws_name, guid_mapping,
                            f"default (variables.json) <- {target_ws_name}",
                            is_default=True, declared_types=declared_types,
                        )
                    else:
                        # Ensure variableOverrides key exists so appends are captured
                        if "variableOverrides" not in data:
                            data["variableOverrides"] = []
                        updates = apply_guid_updates(
                            data["variableOverrides"], ws_id, target_ws_name, guid_mapping,
                            f"{path} <- {target_ws_name}",
                            is_default=False, declared_types=declared_types,
                        )

                    if not updates:
                        results["skipped"].append({
                            "name": target_key,
                            "reason": "already up to date",
                        })
                    elif DRY_RUN:
                        pending_changes += len(updates)
                        results["skipped"].append({
                            "name": target_key,
                            "reason": f"DRY_RUN=True - {len(updates)} change(s) not written",
                        })
                    else:
                        results["succeeded"].append(target_key)
                        # Reserialize only when something actually changed - an
                        # unconditional dumps() reformats the whole part and churns
                        # Git-integrated repos. ensure_ascii=False keeps non-ASCII
                        # note text intact; the trailing newline is preserved.
                        trailing = "\n" if raw.endswith("\n") else ""
                        raw = json.dumps(data, indent=2, ensure_ascii=False) + trailing

                except (KeyError, TypeError, ValueError) as ex:
                    print(f"    ERROR   {target_key}: {ex}")
                    results["failed"].append({
                        "name": target_key,
                        "error": f"apply_guid_updates: {ex}",
                    })
            # else: already recorded as skipped during validation above

        encoded = base64.b64encode(raw.encode("utf-8")).decode("utf-8")
        updated_parts.append(
            {"path": path, "payload": encoded, "payloadType": "InlineBase64"}
        )

    # Targets configured but not present in the definition - flag as skipped
    # so the user sees that their config didn't match any part.
    for target in VALUE_SETS_TO_UPDATE:
        if target not in touched_targets and target not in {
            e["name"] for e in results["skipped"]
        } and target not in {e["name"] for e in results["failed"]}:
            results["skipped"].append({
                "name": target,
                "reason": f"no matching part in definition (expected path 'variables.json' or 'valueSets/{target}.json')",
            })

    # POST updated definition back
    if results["succeeded"]:
        try:
            update_url = (
                f"{FABRIC_API_BASE}/workspaces/{vl_ws_id}"
                f"/VariableLibraries/{vl_item_id}/updateDefinition"
            )
            body = {"definition": {"parts": updated_parts}}
            response = poll_lro(fabric_request("POST", update_url, json_body=body))

            if response.status_code in (200, 202):
                print(
                    f"\nVariable Library '{VARIABLE_LIBRARY_NAME}' updated successfully."
                )
            else:
                # Move every previously-succeeded value set into failed - the
                # API call that would persist their changes didn't land.
                err = f"updateDefinition: {response.status_code} {response.text}"
                for vs_name in list(results["succeeded"]):
                    results["succeeded"].remove(vs_name)
                    results["failed"].append({"name": vs_name, "error": err})
                print(f"\nERROR: Update failed: {response.status_code} {response.text}")
        except (RuntimeError, TimeoutError, requests.RequestException) as ex:
            err = f"updateDefinition: {ex}"
            for vs_name in list(results["succeeded"]):
                results["succeeded"].remove(vs_name)
                results["failed"].append({"name": vs_name, "error": err})
            print(f"\nERROR: Update failed: {ex}")
    elif pending_changes:
        print(
            f"\nDRY_RUN: {pending_changes} pending change(s) computed and NOT written. "
            f"Review the diff above, then set DRY_RUN = False to post it."
        )
    else:
        print("\nAll targets are already up to date or skipped. No update needed.")

# ── Summary ──────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("  Summary")
print("─" * 80)
print(f"  Succeeded: {len(results['succeeded'])}")
print(f"  Skipped:   {len(results['skipped'])}")
print(f"  Failed:    {len(results['failed'])}")

for name in results["succeeded"]:
    print(f"    OK   {name}")
for entry in results["skipped"]:
    print(f"    SKIP {entry['name']}: {entry['reason']}")
for entry in results["failed"]:
    print(f"    FAIL {entry['name']}: {entry['error']}")

print("─" * 80)

if STRICT and results["failed"]:
    raise RuntimeError(
        f"STRICT mode: {len(results['failed'])} target(s) failed. "
        f"See summary above."
    )